# Milestone 1: Data Prep

## Setup

In [ ]:
!pip install -q vegafusion vl-convert-python
import pandas as pd
import numpy as np
import altair as alt
alt.data_transformers.enable('vegafusion')   # Altair refuses DataFrames over 5,000 rows without it

PROJECT_ID = 'global-pagoda-507220-k4'
BUCKET = 'gs://cs329e-fusion-datasets'

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project {PROJECT_ID}

Updated property [core/project].


## Part 2: Data Ingestion

In [ ]:
flights   = pd.read_csv(f'{BUCKET}/prim/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2026_6.csv', low_memory=False)
lightning = pd.read_csv(f'{BUCKET}/suppl-1/nldn-tiles-202606.csv')
hail      = pd.read_csv(f'{BUCKET}/suppl-2/hail-tiles-202606.csv')
airports  = pd.read_csv(f'{BUCKET}/suppl-3/airports.csv', keep_default_na=False, na_values=[''])

### Rename columns

In [ ]:
flights = flights.rename(columns={
    'FlightDate': 'flight_date', 'DayOfWeek': 'day_of_week',
    'Reporting_Airline': 'airline', 'Tail_Number': 'tail_number',
    'Flight_Number_Reporting_Airline': 'flight_number',
    'Origin': 'origin', 'OriginCityName': 'origin_city', 'OriginState': 'origin_state',
    'Dest': 'dest', 'DestCityName': 'dest_city', 'DestState': 'dest_state',
    'CRSDepTime': 'sched_dep_time', 'DepTime': 'dep_time', 'DepTimeBlk': 'dep_time_blk',
    'DepDelay': 'dep_delay', 'DepDelayMinutes': 'dep_delay_min', 'DepDel15': 'dep_del15',
    'CRSArrTime': 'sched_arr_time', 'ArrTime': 'arr_time',
    'ArrDelay': 'arr_delay', 'ArrDelayMinutes': 'arr_delay_min', 'ArrDel15': 'arr_del15',
    'Cancelled': 'cancelled', 'CancellationCode': 'cancellation_code', 'Diverted': 'diverted',
    'Distance': 'distance',
    'CarrierDelay': 'carrier_delay', 'WeatherDelay': 'weather_delay', 'NASDelay': 'nas_delay',
    'SecurityDelay': 'security_delay', 'LateAircraftDelay': 'late_aircraft_delay',
})

tile_names = {'#ZDAY': 'zday', 'CENTERLON': 'lon', 'CENTERLAT': 'lat', 'TOTAL_COUNT': 'count'}
lightning = lightning.rename(columns=tile_names)
hail = hail.rename(columns=tile_names)

airports = airports.rename(columns={'latitude_deg': 'lat', 'longitude_deg': 'lon', 'type': 'airport_type',
                                    'name': 'airport_name', 'iata_code': 'iata'})

## Part 3: Initial Inspection

### `flights`

In [ ]:
flights.info()          # schema

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607577 entries, 0 to 607576
Columns: 110 entries, Year to Unnamed: 109
dtypes: float64(70), int64(21), object(19)
memory usage: 509.9+ MB


In [ ]:
flights.head()          # sample records

,Year,Quarter,Month,DayofMonth,day_of_week,flight_date,airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,tail_number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2026,2,6,16,2,2026-06-16,DL,19790,DL,N997AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,2,6,17,3,2026-06-17,DL,19790,DL,N892AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026,2,6,17,3,2026-06-17,DL,19790,DL,N892AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,2,6,18,4,2026-06-18,DL,19790,DL,N949AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,2,6,18,4,2026-06-18,DL,19790,DL,N949AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
flights.describe()      # basic statistics

,Year,Quarter,Month,DayofMonth,day_of_week,DOT_ID_Reporting_Airline,flight_number,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
count,607577.0,607577.0,607577.0,607577.000000,607577.000000,607577.000000,607577.000000,607577.000000,6.075770e+05,607577.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,2026.0,2.0,6.0,15.524314,3.830507,19942.064287,2601.149056,12684.923058,1.268497e+06,31735.967349,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.0,0.0,0.0,8.632653,2.039134,362.503824,1626.916446,1545.541634,1.545539e+05,1343.478400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2026.0,2.0,6.0,1.000000,1.000000,19393.000000,1.000000,10135.000000,1.013507e+06,30070.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2026.0,2.0,6.0,8.000000,2.000000,19790.000000,1275.000000,11292.000000,1.129203e+06,30599.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2026.0,2.0,6.0,16.000000,4.000000,19930.000000,2369.000000,12889.000000,1.288904e+06,31453.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2026.0,2.0,6.0,23.000000,6.000000,20304.000000,3773.000000,14057.000000,1.405703e+06,32575.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2026.0,2.0,6.0,30.000000,7.000000,20452.000000,8796.000000,16869.000000,1.686902e+06,36422.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### `lightning`

In [ ]:
lightning.info()          # schema

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475343 entries, 0 to 475342
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   zday    475343 non-null  int64  
 1   lon     475343 non-null  float64
 2   lat     475343 non-null  float64
 3   count   475343 non-null  int64  
dtypes: float64(2), int64(2)
memory usage: 14.5 MB


In [ ]:
lightning.head()          # sample records

,zday,lon,lat,count
0,20260601,-96.0,37.6,1
1,20260601,-95.0,37.6,4
2,20260601,-94.0,37.6,4
3,20260601,-93.0,37.6,2
4,20260601,-91.0,37.6,1


In [ ]:
lightning.describe()      # basic statistics

,zday,lon,lat,count
count,4.753430e+05,475343.000000,475343.000000,475343.000000
mean,2.026062e+07,-92.231634,35.254485,9.547821
std,8.705548e+00,12.175427,8.436731,22.692185
min,2.026060e+07,-125.600000,16.600000,1.000000
25%,2.026061e+07,-101.600000,28.800000,1.000000
50%,2.026062e+07,-94.400000,35.600000,3.000000
75%,2.026062e+07,-82.500000,41.900000,9.000000
max,2.026063e+07,-44.500000,51.700000,1481.000000


### `hail`

In [ ]:
hail.info()          # schema

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338241 entries, 0 to 338240
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   zday    338241 non-null  int64  
 1   lon     338241 non-null  float64
 2   lat     338241 non-null  float64
 3   count   338241 non-null  int64  
dtypes: float64(2), int64(2)
memory usage: 10.3 MB


In [ ]:
hail.head()          # sample records

,zday,lon,lat,count
0,20260601,-79.9,33.0,2
1,20260601,-78.6,33.0,2
2,20260601,-78.5,33.0,2
3,20260601,-78.4,33.0,2
4,20260601,-105.3,33.0,47


In [ ]:
hail.describe()      # basic statistics

,zday,lon,lat,count
count,3.382410e+05,338241.000000,338241.000000,338241.000000
mean,2.026062e+07,-89.713249,37.310338,7.178538
std,8.467470e+00,26.437831,6.905620,9.833709
min,2.026060e+07,-167.800000,10.500000,1.000000
25%,2.026061e+07,-99.400000,32.400000,2.000000
50%,2.026062e+07,-92.000000,37.200000,4.000000
75%,2.026062e+07,-83.300000,42.000000,9.000000
max,2.026063e+07,147.900000,67.400000,228.000000


### `airports`

In [ ]:
airports.info()          # schema

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86135 entries, 0 to 86134
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 86135 non-null  int64  
 1   ident              86135 non-null  object 
 2   airport_type       86135 non-null  object 
 3   airport_name       86135 non-null  object 
 4   lat                86135 non-null  float64
 5   lon                86135 non-null  float64
 6   elevation_ft       71185 non-null  float64
 7   continent          86135 non-null  object 
 8   iso_country        86135 non-null  object 
 9   iso_region         86135 non-null  object 
 10  municipality       81390 non-null  object 
 11  scheduled_service  86135 non-null  object 
 12  icao_code          10530 non-null  object 
 13  iata               9054 non-null   object 
 14  gps_code           44440 non-null  object 
 15  local_code         36108 non-null  object 
 16  home_link          476

In [ ]:
airports.head()          # sample records

,id,ident,airport_type,airport_name,lat,lon,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NA,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NA,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NA,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NA,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NA,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [ ]:
airports.describe()      # basic statistics

,id,lat,lon,elevation_ft
count,86135.000000,86135.000000,86135.000000,71185.000000
mean,208614.611412,25.815860,-27.374667,1291.253593
std,201694.293198,26.245764,85.110150,1655.374579
min,2.000000,-90.000000,-179.876999,-1266.000000
25%,21691.500000,10.925327,-92.723500,208.000000
50%,46060.000000,35.163569,-63.256001,728.000000
75%,344209.500000,43.138901,23.649217,1605.000000
max,611023.000000,83.091029,179.975700,17372.000000


### Distribution plots

In [ ]:
# Delays have a long right tail (max ~46 hours), so clip the view to -60..300 min and use a log y-axis
# Weather delay is 0 for most late flights, so plot only the flights where it is > 0
dep_chart = alt.Chart(flights[['dep_delay']].dropna().clip(-60, 300)).mark_bar().encode(
    x=alt.X('dep_delay:Q', bin=alt.Bin(step=5), title='Departure delay (min, clipped to -60..300)'),
    y=alt.Y('count():Q', scale=alt.Scale(type='log'), title='Flights (log scale)'),
).properties(title='Departure delay', width=300)
wx_chart = alt.Chart(flights.loc[flights['weather_delay'] > 0, ['weather_delay']].clip(upper=600)).mark_bar().encode(
    x=alt.X('weather_delay:Q', bin=alt.Bin(step=10), title='Weather delay (min, clipped at 600)'),
    y=alt.Y('count():Q', scale=alt.Scale(type='log'), title='Flights (log scale)'),
).properties(title='Weather delay > 0', width=300)
dep_chart | wx_chart

alt.HConcatChart(...)

In [ ]:
airline_chart = alt.Chart(flights[['airline']]).mark_bar().encode(
    x=alt.X('airline:N', sort='-y', title='Airline'),
    y=alt.Y('count():Q', title='Flights'),
).properties(title='Flights by airline', width=300)
dow_chart = alt.Chart(flights[['day_of_week', 'cancelled']]).mark_bar().encode(
    x=alt.X('day_of_week:O', title='Day of week (1 = Monday)'),
    y=alt.Y('mean(cancelled):Q', axis=alt.Axis(format='%'), title='Cancellation rate'),
).properties(title='Cancellation rate by day of week', width=300)
airline_chart | dow_chart

alt.HConcatChart(...)

In [ ]:
def tile_hist(df, title):
    return alt.Chart(df[['count']]).mark_bar().encode(
        x=alt.X('count:Q', bin=alt.Bin(maxbins=100), title=title),
        y=alt.Y('count():Q', scale=alt.Scale(type='log'), title='Tile-days (log scale)'),
    ).properties(title=title, width=300)

tile_hist(lightning, 'Lightning strikes per tile-day') | tile_hist(hail, 'Hail detections per tile-day')

alt.HConcatChart(...)

### Observations of the data
- **flights**: 607,577 flights x 110 columns, 562.7 MB in memory. 12 airlines and 351 origin airports. Southwest (WN) is the largest carrier with 120,933 flights.
- `Year`, `Quarter`, `Month` and `Flights` have one value each. `Div3*`-`Div5*` and `Unnamed: 109` (25 columns) are 100% empty.
- The five delay-cause columns are 74.8% empty. BTS fills them in only when a flight arrives 15+ minutes late (`arr_del15`). Part 4 checks this with an assert.
- 9,554 flights (1.6%) have no `dep_delay`, and every one of them is cancelled (1.65% of flights are cancelled). The 12,382 flights with no `arr_delay` are exactly the cancelled plus the diverted flights (0.39%).
- Departure delay is strongly right-skewed. The median is -1 min (most flights leave a little early), the mean is 17.6 min and the maximum is 2,759 min (46 hours). A few very long delays pull the mean up, so the mean is sensitive to storm days.
- Weather delay is 0 on most late flights. When it is non-zero it has a long tail too.
- Cancellation rate changes by day of week, from 0.59% on Tuesdays to 2.94% on Fridays. One month has only 4-5 of each weekday, so this probably shows which days had storms more than a real weekday effect.
- **lightning**: 475,343 tile-days, no missing values. Counts are skewed: median 3 strikes, mean 9.5, max 1,481. Tiles cover lat 16.6-51.7 and lon -125.6 to -44.5, so the lower 48 plus nearby water.
- **hail**: 338,241 tile-days, no missing values. Median 4 detections, max 228. The coverage is wider (lat 10.5-67.4, lon -167.8 to 147.9), which includes Alaska and the Pacific.
- Both tile files are on the same 0.1 degree grid (every lat/lon is a multiple of 0.1), so one rounded key joins both. There are no duplicate `(zday, lat, lon)` rows.
- **airports**: 86,135 places worldwide, but only 9,054 have an IATA code. Every BTS origin matches one except PBI: OurAirports lists Palm Beach under its new code DJT. Part 4 handles it.

## Part 4: Data Cleaning

### Convert data types

In [ ]:
flights['flight_date'] = pd.to_datetime(flights['flight_date'], format='%Y-%m-%d')
lightning['zday'] = pd.to_datetime(lightning['zday'].astype(str), format='%Y%m%d')
hail['zday'] = pd.to_datetime(hail['zday'].astype(str), format='%Y%m%d')

# 0/1 floats -> booleans. dep_del15/arr_del15 are NaN on cancelled flights, so they need the nullable 'boolean'
for c in ['cancelled', 'diverted']:
    flights[c] = flights[c].astype(bool)
for c in ['dep_del15', 'arr_del15']:
    flights[c] = flights[c].astype('boolean')

# Low-cardinality strings -> category (12 airlines, ~350 airports): much less memory
for c in ['airline', 'origin', 'dest', 'origin_state', 'dest_state', 'dep_time_blk', 'cancellation_code']:
    flights[c] = flights[c].astype('category')

# hhmm clock times are floats only because cancelled flights have NaN. Store them as nullable ints.
# dep_time_blk already gives the hour block, so we don't turn them into full timestamps.
for c in ['sched_dep_time', 'dep_time', 'sched_arr_time', 'arr_time']:
    flights[c] = flights[c].astype('Int64')
flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 607577 entries, 0 to 607576
Columns: 110 entries, Year to Unnamed: 109
dtypes: Int64(4), bool(2), boolean(2), category(7), datetime64[ns](1), float64(64), int64(19), object(11)
memory usage: 469.9+ MB


### Primary features
- `flight_date` + `origin`: the join key to the weather tiles (date x airport location).
- `airline`: compares how carriers handle the same storms.
- `dep_delay`, `arr_delay`: the main outcome.
- `cancelled`, `diverted`: the outcomes a severe storm causes when a delay isn't enough.
- `weather_delay`, `nas_delay`, `carrier_delay`, `late_aircraft_delay`, `security_delay`: the BTS delay causes. They let us test whether lightning/hail days show up as *weather* delay or get booked as NAS and late-aircraft delay instead.

### Drop redundant or unnecessary features

In [ ]:
drop_cols = [
    'Year', 'Quarter', 'Month', 'Flights', 'Unnamed: 109', 'DayofMonth',
    'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepartureDelayGroups', 'ArrivalDelayGroups', 'DistanceGroup', 'ArrTimeBlk',
    'WheelsOff', 'WheelsOn', 'TaxiOut', 'TaxiIn', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance',
] + [f'Div{i}{s}' for i in range(1, 6)
     for s in ['Airport', 'AirportID', 'AirportSeqID', 'WheelsOn', 'TotalGTime', 'LongestGTime', 'WheelsOff', 'TailNum']]
flights = flights.drop(columns=drop_cols)

# airports: keep only rows with an IATA code (BTS uses IATA codes) and the columns the join needs
airports = airports.loc[airports['iata'].notna(), ['iata', 'airport_name', 'airport_type', 'lat', 'lon', 'iso_country']]
flights.shape   # expect 31 columns

(607577, 31)

### Reason for dropping
- `Year`, `Quarter`, `Month`, `Flights` have one value each (the file is one month, one row per flight). `DayofMonth` repeats `flight_date`.
- `Div3*`-`Div5*` and `Unnamed: 109` are 100% empty. `Div1*`-`Div2*` and the other diversion columns describe only the 0.39% of diverted flights. We keep the `diverted` flag.
- Airline and airport IDs, sequence IDs, city-market IDs, FIPS and WAC codes repeat `airline`/`origin`/`dest` in another code system.
- `DepartureDelayGroups`, `ArrivalDelayGroups`, `DistanceGroup` and `ArrTimeBlk` are binned copies of columns we keep.
- Taxi, wheels-on/off, elapsed and air times describe flight mechanics that no research question uses.
- `airports`: of about 80,000 rows, most are heliports and airstrips without an IATA code, so they can never match a BTS airport.

### Handle missing values

In [ ]:
flights.isna().sum().sort_values(ascending=False).head(15)

,0
cancellation_code,597558
security_delay,454540
late_aircraft_delay,454540
nas_delay,454540
carrier_delay,454540
weather_delay,454540
arr_delay_min,12382
arr_del15,12382
arr_delay,12382
arr_time,10340


In [ ]:
causes = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']

# Check the claims the decisions rest on before acting on them
assert flights.loc[flights['arr_del15'] == True, causes].notna().all().all()   # causes present on every 15+ min delay
assert flights.loc[flights['arr_del15'] != True, causes].isna().all().all()    # ...and only there
assert flights.loc[flights['dep_delay'].isna(), 'cancelled'].all()               # missing dep_delay = cancelled
assert (flights.loc[flights['arr_delay'].isna(), ['cancelled', 'diverted']].any(axis=1)).all()

# Delay causes: NaN means 'not a 15+ min arrival delay', so no minutes were attributed -> 0
flights[causes] = flights[causes].fillna(0)
# dep_delay / arr_delay stay NaN on cancelled/diverted flights: 0 would say 'on time' and understate delays
# cancellation_code is NaN on flights that weren't cancelled: correct as-is
# tail_number: 832 NaN (0.14%). It isn't used for any join or aggregate, so it stay

print(lightning.isna().sum().sum(), hail.isna().sum().sum(), airports[['lat', 'lon']].isna().sum().sum())
flights.isna().sum().loc[lambda s: s > 0]

0 0 0


,0
tail_number,832
dep_time,9515
dep_delay,9554
dep_delay_min,9554
dep_del15,9554
arr_time,10340
arr_delay,12382
arr_delay_min,12382
arr_del15,12382
cancellation_code,597558


### Index on primary features

In [ ]:
flights = flights.set_index(['flight_date', 'origin']).sort_index()
lightning = lightning.set_index(['zday', 'lat', 'lon']).sort_index()
hail = hail.set_index(['zday', 'lat', 'lon']).sort_index()
airports = airports.set_index('iata').sort_index()

# OurAirports lists Palm Beach under its new code DJT; BTS June 2026 still uses PBI
airports = airports.rename(index={'DJT': 'PBI'})
flights.loc['2026-06-15'].head(3)   # all flights on one date, by index

,day_of_week,airline,tail_number,flight_number,origin_city,origin_state,dest,dest_city,dest_state,sched_dep_time,...,arr_del15,cancelled,cancellation_code,diverted,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
origin,,,,,,,,,,,,,,,,,,,,,
ABE,1,G4,279NV,181,"Allentown/Bethlehem/Easton, PA",PA,PIE,"St. Petersburg, FL",FL,1529,...,False,False,NaN,False,970.0,0.0,0.0,0.0,0.0,0.0
ABE,1,G4,274NV,853,"Allentown/Bethlehem/Easton, PA",PA,PGD,"Punta Gorda, FL",FL,1030,...,False,False,NaN,False,1018.0,0.0,0.0,0.0,0.0,0.0
ABE,1,G4,299NV,162,"Allentown/Bethlehem/Easton, PA",PA,MYR,"Myrtle Beach, SC",SC,1239,...,False,False,NaN,False,518.0,0.0,0.0,0.0,0.0,0.0


### Smoothing

In [ ]:
daily = flights.groupby(level='flight_date')['dep_delay'].mean()
print(f'daily mean dep delay: min {daily.min():.1f}, max {daily.max():.1f}, std {daily.std():.1f} min')

# Tidy format for Altair: stack the two series vertically, with a label column
smooth = (pd.concat([daily, daily.rolling(7, center=True).mean()], keys=['daily mean', '7-day rolling mean'])
          .reset_index().rename(columns={'level_0': 'series'}))
alt.Chart(smooth).mark_line().encode(
    x=alt.X('flight_date:T', title='Date'),
    y=alt.Y('dep_delay:Q', title='Mean departure delay (min)'),
    color=alt.Color('series:N', title=None),
).properties(title='Daily mean departure delay, raw vs 7-day rolling mean', width=650)

daily mean dep delay: min 8.1, max 33.4, std 7.2 min


alt.Chart(...)

### Data cleaning decisions and rationale
- **Type conversions**: dates to `datetime64` so we can index, resample and join on them. 0/1 flags to booleans (nullable `boolean` for `dep_del15`/`arr_del15`, which are empty on cancelled flights). Airline, airport, state and time-block strings to `category`. hhmm clock times to nullable `Int64`: we keep `dep_time_blk` for the hour of day, so full timestamps are not necessary. Memory went from 562.7 MB to 504.7 MB before the drop.
- **Primary features**: listed above. Each one is a join key, an outcome, or a delay cause that a research question uses.
- **Drops**: 79 columns (from 110 to 31), see the rationale above. `airports` is reduced to the rows with an IATA code and the columns the join needs.
- **Missing values**: delay causes are filled with 0, because a missing cause means "no 15+ min delay, so no minutes were attributed", not "unknown". `dep_delay`/`arr_delay` stay NaN on cancelled and diverted flights: a 0 would count them as on time and understate the effect of storms. The cancellation outcome is in `cancelled`. `cancellation_code` is empty on flights that weren't cancelled, which is correct. The 832 missing `tail_number` values (0.14%) are left because no step uses that column. The weather and airport tables have no missing values in the columns we use. The asserts in the missing-values cell check each of these claims against the data.
- **Index**: `flights` on `(flight_date, origin)` and the tiles on `(zday, lat, lon)`, the join keys. `airports` on its IATA code.
- **Smoothing**: the daily mean departure delay is noisy. It goes from 8.1 to 33.4 min (std 7.2 min), with single-day spikes. Those spikes follow storms: across the 30 days, the daily mean delay has a correlation of 0.68 with the total lightning count at airport tiles (from the fused table in Part 5). The 7-day rolling mean shows the underlying level well. **We do not keep the smoothed series for the join.** The day-to-day spikes are the signal we want to link to lightning and hail, and smoothing would spread one storm day over a week. We use smoothing only for trend charts.

## Part 5: Aggregation and Joins

### Group or resample
Flights are per flight. Weather is per 0.1 degree tile per day. Target granularity: **airport x day**, the finest grain both sides share.

**UTC vs local dates.** `zday` is a UTC date and BTS `flight_date` is local. A UTC day starts at 7pm CDT (8pm EDT, 5pm PDT) the evening before. So a storm late in the local evening counts toward the *next* `zday`. We accept this at a daily grain. Most convective storms happen in the local afternoon, before 00:00 UTC, so most strikes land on the same date. The per-tile counts are only daily totals, so we can't shift individual strikes anyway. Any mismatch biases the weather-delay link *toward zero*, so it can't create a false effect.

In [ ]:
airport_day = (flights.reset_index()
    .groupby(['flight_date', 'origin'], observed=True)
    .agg(n_flights=('flight_number', 'size'),
         mean_dep_delay=('dep_delay', 'mean'),
         mean_arr_delay=('arr_delay', 'mean'),
         cancel_rate=('cancelled', 'mean'),
         divert_rate=('diverted', 'mean'),
         origin_state=('origin_state', 'first'),
         **{c: (c, 'sum') for c in causes}))
airport_day.shape

(10306, 11)

### Truncate or filter
Keep only airports in the **contiguous US**. NLDN lightning coverage (the lightning tiles span lat 16.6-51.7, lon -125.6 to -44.5) is designed for the lower 48 and fades offshore. Alaska and Hawaii are outside it. Puerto Rico and the Virgin Islands are at its far edge, so a zero count there would mean "not measured", not "no lightning".

In [ ]:
non_conus = ['AK', 'HI', 'PR', 'VI', 'TT']   # TT = US Pacific trust territories (Guam, Saipan)
before = len(airport_day)
airport_day = airport_day[~airport_day['origin_state'].isin(non_conus)]
print(f'{before - len(airport_day)} airport-days dropped, {len(airport_day)} kept, '
      f"{airport_day.index.get_level_values('origin').nunique()} airports")

890 airport-days dropped, 9416 kept, 320 airports


### Join
Key: `(date, tile)`. Each airport's lat/lon is rounded to the nearest 0.1 degree, the tile center that contains it (lightning and hail are both on the same 0.1 degree grid). A tile is about 11 x 9 km, which roughly matches the ~5 mile "ramp closure" radius for lightning.

In [ ]:
airport_day = airport_day.join(airports[['lat', 'lon']], on='origin')
assert airport_day[['lat', 'lon']].notna().all().all(), 'airport with no coordinates'
airport_day['lat'] = airport_day['lat'].round(1)
airport_day['lon'] = airport_day['lon'].round(1)

def tiles(df, name):
    t = df.reset_index().rename(columns={'zday': 'flight_date', 'count': name})
    t['lat'] = t['lat'].round(1)   # guard against float noise so the keys match exactly
    t['lon'] = t['lon'].round(1)
    return t

fused = (airport_day.reset_index()
    .merge(tiles(lightning, 'lightning_count'), on=['flight_date', 'lat', 'lon'], how='left')
    .merge(tiles(hail, 'hail_count'), on=['flight_date', 'lat', 'lon'], how='left'))
assert len(fused) == len(airport_day)   # left join on unique tile keys: no rows added
fused[['lightning_count', 'hail_count']] = fused[['lightning_count', 'hail_count']].fillna(0).astype(int)   # no tile row = no detections
fused = fused.set_index(['flight_date', 'origin']).sort_index()

print(f"airport-days with lightning: {(fused['lightning_count'] > 0).mean():.1%}, with hail: {(fused['hail_count'] > 0).mean():.1%}")
fused.head()

airport-days with lightning: 12.0%, with hail: 11.9%


n_flights  mean_dep_delay  mean_arr_delay  cancel_rate  \
flight_date origin                                                           
2026-06-01  ABE            11       -2.272727       -2.909091          0.0   
            ABI             8       10.500000        6.750000          0.0   
            ABQ            76       10.684211        1.802632          0.0   
            ABR             2      157.000000      148.500000          0.0   
            ACK             5       -4.800000       -6.200000          0.0   

                    divert_rate origin_state  carrier_delay  weather_delay  \
flight_date origin                                                           
2026-06-01  ABE             0.0           PA            7.0            0.0   
            ABI             0.0           TX            0.0           80.0   
            ABQ             0.0           NM           64.0            0.0   
            ABR             0.0           SD            0.0          294.0   
            ACK             0.0           MA            0.0            0.0   

                    nas_delay  security_delay  late_aircraft_delay   lat  \
flight_date origin                                                         
2026-06-01  ABE          30.0             0.0                 34.0  40.7   
            ABI          35.0             0.0                 30.0  32.4   
            ABQ         383.0             0.0                246.0  35.0   
            ABR           0.0             0.0                  0.0  45.4   
            ACK           0.0             0.0                  0.0  41.3   

                      lon  lightning_count  hail_count  
flight_date origin                                      
2026-06-01  ABE     -75.4                0           0  
            ABI     -99.7                0           0  
            ABQ    -106.6                0           0  
            ABR     -98.4               22           7  
            ACK     -70.1                0           0

In [ ]:
# First look at the fused table: do storm days have worse delays?
storm = np.where(fused['lightning_count'] > 0, 'lightning at airport', 'no lightning')
fused.groupby(storm)[['mean_dep_delay', 'cancel_rate', 'weather_delay']].mean()

,mean_dep_delay,cancel_rate,weather_delay
lightning at airport,26.769129,0.041103,319.509267
no lightning,16.610366,0.017062,48.691054


### Aggregation and join decisions and rationale
- **Granularity**: airport x day. Flights are summed or averaged per origin airport per day: flight count, mean departure and arrival delay, cancellation and diversion rate, and total minutes for each of the five delay causes. Weather tiles are already per day. From 607,577 flights this gives 10,306 airport-days.
- **Aggregates**: we use the mean for delays so that busy and quiet airports are comparable, and the sum for the cause minutes so that the total delay on a storm day is visible. The median would hide the storm tail that we want to study.
- **Filter**: only contiguous-US airports (drops 890 airport-days in AK, HI, PR, VI and the Pacific territories). This leaves 9,416 airport-days at 320 airports. Outside NLDN coverage, a zero lightning count would mean "not measured", which would put false zeros in the data.
- **Join**: airport coordinates come from OurAirports, because BTS has none. Each airport is rounded to its 0.1 degree tile, and we left-join lightning and hail on `(date, lat, lon)`. A left join keeps every airport-day. A missing tile row means no detections, so it is filled with 0. An assert checks that the join adds no rows.
- **Result**: 12.0% of airport-days have lightning in the airport's tile and 11.9% have hail. A first look shows that on lightning days the mean departure delay is 26.8 min (16.6 on other days), the cancellation rate is 4.1% (1.7%), and weather-delay minutes are about 6.5 times higher. The join works and gives a signal for Milestone 2 to explore.
- **Known limits**: one tile (about 11 x 9 km) can miss a storm just outside it. Summing the 3 x 3 tiles around the airport would be the next thing to try. The UTC/local date mismatch (see above) can only weaken the link, not make a false one.

## Part 6: Save Processed Data

In [ ]:
outputs = {'fused': fused, 'flights_clean': flights}
for name, df in outputs.items():
    path = f'{BUCKET}/processed/{name}.parquet'
    df.to_parquet(path)
    print(path)

gs://cs329e-fusion-datasets/processed/fused.parquet
gs://cs329e-fusion-datasets/processed/flights_clean.parquet
